# Calibrated Equalized Odds on credit default data

Runs the full pipeline on all three datasets and produces every table and figure used in the README.

**Method:** Calibrated Equalized Odds post-processing (Pleiss et al., *On Fairness and Calibration*, NeurIPS 2017), via AIF360's `CalibratedEqOddsPostprocessing`.
**Datasets:** German Credit (protected: age, older than 25 = privileged), Taiwan credit card default (sex, male = privileged), Home Credit Default Risk (sex, male = privileged). Label 1 = repays (favourable), so a positive prediction is an approval.

**Per dataset and seed:** fairness of the original labels → stratified 60/20/20 split on label × group → train the base model (missing values imputed inside the model, on the training split only) → choose the cutoff on validation (max TPR − FPR) → fit the post-processor on validation, apply it to test → evaluate baseline and mitigated predictions on test.

## Settings

In [ ]:
DATASETS = ["german", "taiwan", "homecredit"]
MODEL = "lr"            # "lr" = logistic regression, "gbm" = gradient boosting with isotonic calibration
CONSTRAINT = "fnr"      # generalised error rate to equalise: "fnr", "fpr" or "weighted"
SEEDS = 5               # number of random stratified splits
FIG_DIR = "../figures"  # tables and plots are saved here

In [ ]:
import sys, logging, warnings
from pathlib import Path
sys.path.insert(0, "../src")
logging.getLogger().setLevel(logging.ERROR)   # hide AIF360's optional-dependency notices
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import Image, display

from fair_credit.data import load
from fair_credit.metrics import label_metrics
from fair_credit.pipeline import run_once
from fair_credit.plots import reliability

Path(FIG_DIR).mkdir(exist_ok=True)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

## 1. Run the pipeline

Home Credit takes a few minutes; German and Taiwan are quick.

In [ ]:
label_rows, run_rows, datasets = [], [], {}
for name in DATASETS:
    ds = load(name)
    datasets[name] = ds
    print(f"{name}: {len(ds.y):,} rows, {ds.X.shape[1]} features, protected = {ds.protected} (privileged: {ds.privileged_desc})")
    label_rows.append({"dataset": name, **label_metrics(ds.X.to_numpy(), ds.y, ds.s)})
    for seed in range(SEEDS):
        base, mitig, curves = run_once(ds, MODEL, CONSTRAINT, seed)
        run_rows += [{"dataset": name, "seed": seed, "stage": "baseline", **base},
                     {"dataset": name, "seed": seed, "stage": "ceo", **mitig}]
        if seed == 0:
            reliability(curves, f"{name}, {MODEL}, constraint = {CONSTRAINT}",
                        Path(FIG_DIR) / f"{name}_{MODEL}_{CONSTRAINT}_reliability.png")

runs = pd.DataFrame(run_rows)
runs.to_csv(Path(FIG_DIR) / f"runs_{MODEL}_{CONSTRAINT}.csv", index=False)

## 2. Fairness of the original labels

DP_Diff = P(y=1 | unprivileged) − P(y=1 | privileged), fair at 0. DIR = their ratio, fair at 1 (0.8 to 1.25 is the usual bound). Consistency compares each person's label with their 5 nearest neighbours on standardised features.

`consistency (full data)` uses all rows. Consistency of predictions can only be measured on the test set, where neighbourhoods are sparser, so `consistency (test rows)` gives the label consistency on exactly those rows: that is the like-for-like reference for Table 2.

In [ ]:
labels = pd.DataFrame(label_rows).set_index("dataset").rename(columns={"consistency": "consistency (full data)"})
labels["consistency (test rows)"] = runs[runs.stage == "baseline"].groupby("dataset")["consistency_labels_test"].mean()
labels.to_csv(Path(FIG_DIR) / "original_labels.csv")
labels

## 3. Before and after mitigation (test set)

Mean over the seeds, standard deviation in brackets. `n` shows on how many seeds a mean is based (DIR and the default rate among approved are undefined when a group has no approvals). Means hide a lot when the method behaves differently per seed, so check Section 5 before reading these as averages.

In [ ]:
METRICS = ["dp_diff", "dir", "consistency", "balanced_accuracy", "auc", "accuracy",
           "gfnr_gap", "gfpr_gap", "ece_priv", "ece_unpriv",
           "approval_priv", "approval_unpriv", "default_among_approved_priv", "default_among_approved_unpriv",
           "threshold"]

def fmt(x):
    x = x.dropna()
    s = f"{x.mean():.3f} ({x.std():.3f})"
    return s if len(x) == SEEDS else s + f" n={len(x)}"

table = (runs.melt(id_vars=["dataset", "seed", "stage"], value_vars=METRICS, var_name="metric")
             .groupby(["metric", "dataset", "stage"], sort=False)["value"].apply(fmt)
             .unstack(["dataset", "stage"]).reindex(METRICS))
table.to_csv(Path(FIG_DIR) / f"before_after_{MODEL}_{CONSTRAINT}.csv")
table

## 4. Did the method reach its own target?

The method equalises a *generalised* error rate, computed from scores. For `fnr` that is the mean of 1 − score over people who repaid. The targeted gap should shrink towards 0; the other gap is free to grow.

In [ ]:
target, other = ("gfnr_gap", "gfpr_gap") if CONSTRAINT == "fnr" else ("gfpr_gap", "gfnr_gap")
gaps = runs.groupby(["dataset", "stage"], sort=False)[[target, other]].mean().unstack("stage").round(4)
gaps

## 5. Per seed: feasibility, mixing rate and outcome

Computed on the validation set, where the post-processor is fitted. The degraded group is moved towards its constant base-rate classifier, the worst *calibrated* classifier it can have.

- `headroom` = trivial cost − current cost of the degraded group. A calibrated model always has headroom > 0. Headroom ≤ 0 means the model is not calibrated for that group on this measure, and the method's assumption breaks.
- `feasible`: the other group's cost is at most the degraded group's trivial cost (Section 4 of the paper).
- `mix rate`: share of the degraded group whose score is replaced by the base rate. AIF360 does not clip it: above 1 means everyone is replaced, below 0 means nobody is.

In [ ]:
ceo = runs[runs.stage == "ceo"].copy()
ceo["degraded group"] = np.where(ceo["val_degraded_is_priv"] == 1, "privileged", "unprivileged")
ceo["mix rate"] = np.where(ceo["val_degraded_is_priv"] == 1, ceo["mix_rate_priv"], ceo["mix_rate_unpriv"])
ceo["what happened"] = np.select([ceo["mix rate"] <= 0, ceo["mix rate"] >= 1],
                                 ["nobody replaced", "everyone replaced"], "part replaced")
per_seed = ceo.set_index(["dataset", "seed"])[[
    "degraded group", "val_cost_degraded", "val_trivial_cost_degraded", "val_headroom",
    "val_cost_other", "val_feasible", "mix rate", "what happened",
    "dp_diff", "dir", "approval_priv", "approval_unpriv", "threshold"]].rename(columns={
    "val_cost_degraded": "cost", "val_trivial_cost_degraded": "trivial cost", "val_headroom": "headroom",
    "val_cost_other": "other group cost", "val_feasible": "feasible"})
per_seed["feasible"] = per_seed["feasible"].map({1.0: "yes", 0.0: "no"})
per_seed.to_csv(Path(FIG_DIR) / f"per_seed_{MODEL}_{CONSTRAINT}.csv")
per_seed.round(4)

Summary per dataset: how often each outcome occurred, and the spread of DP_Diff after mitigation.

In [ ]:
summary = pd.DataFrame({
    "feasible seeds": per_seed["feasible"].eq("yes").groupby("dataset").sum(),
    "min headroom": per_seed["headroom"].groupby("dataset").min(),
    "nobody replaced": per_seed["what happened"].eq("nobody replaced").groupby("dataset").sum(),
    "part replaced": per_seed["what happened"].eq("part replaced").groupby("dataset").sum(),
    "everyone replaced": per_seed["what happened"].eq("everyone replaced").groupby("dataset").sum(),
    "DP_Diff after, min": per_seed["dp_diff"].groupby("dataset").min(),
    "DP_Diff after, max": per_seed["dp_diff"].groupby("dataset").max(),
}).reindex(DATASETS)
summary.to_csv(Path(FIG_DIR) / f"outcome_summary_{MODEL}_{CONSTRAINT}.csv")
summary.round(4)

## 6. Calibration per group, before and after (seed 0)

Points on the diagonal mean the scores are calibrated for that group. A group whose scores were replaced by its base rate collapses to a few points. Section 3 also reports the expected calibration error (ECE) per group.

In [ ]:
for name in DATASETS:
    display(Image(filename=str(Path(FIG_DIR) / f"{name}_{MODEL}_{CONSTRAINT}_reliability.png")))